# 03 - Supervised Machine Learning Model Training & Evaluation

This notebook trains and evaluates the baseline Logistic Regression and Random Forest classifiers using a chronological time-based split and required performance metrics.

In [1]:
import os, sys, pickle
import pandas as pd
import numpy as np
sys.path.insert(0, os.path.abspath('..'))
from config.config import RAW_DATA_PATH, MODELS_DIR
from src.models.train import evaluate_classifier, run_training

metrics_list = run_training()
results_df = pd.DataFrame(metrics_list)
results_df[['model', 'precision', 'recall', 'f1_score', 'pr_auc', 'fp_per_1000', 'total_cost']]

[Training] Loading raw dataset from /Users/yaseen/Data_Analysis/BDP Project/data/raw/creditcard.csv...


 -> Total Records: 284807
 -> Legitimate: 284,315 (99.827%)
 -> Fraud Cases: 492 (0.173%) [Extreme Class Imbalance]
 -> Missing values found: 0

[Time-Based Split] Split cutoff time: 145247.0s
 -> Train set: 227,845 txns | Fraud: 417 (0.183%)
 -> Test set:  56,962 txns | Fraud: 75 (0.132%)

[Model 1/3] Training Logistic Regression (Baseline)...



[Model 2/3] Training Random Forest Classifier...



[Model 3/3] Training Isolation Forest (Unsupervised Anomaly Detection)...



========================= MODEL EVALUATION RESULTS =========================
              model  precision  recall  f1_score  pr_auc  fp_per_1000  total_cost
Logistic Regression     0.0441  0.9067    0.0842  0.7622        25.86     25595.0
      Random Forest     0.9194  0.7600    0.8321  0.8150         0.09      9075.0
   Isolation Forest     0.0000  0.0000    0.0000  0.0319         1.69     38940.0

[Artifacts] Successfully saved models to /Users/yaseen/Data_Analysis/BDP Project/models:
 -> logistic_regression.pkl
 -> random_forest.pkl
 -> isolation_forest.pkl
 -> scaler.pkl


,model,precision,recall,f1_score,pr_auc,fp_per_1000,total_cost
0,Logistic Regression,0.0441,0.9067,0.0842,0.7622,25.86,25595.0
1,Random Forest,0.9194,0.7600,0.8321,0.8150,0.09,9075.0
2,Isolation Forest,0.0000,0.0000,0.0000,0.0319,1.69,38940.0


In [2]:
# Inspect saved artifacts
print('Saved model artifacts in models/:')
for f in os.listdir(MODELS_DIR):
    if f.endswith('.pkl'):
        print(' ->', f, f'({os.path.getsize(os.path.join(MODELS_DIR, f)) / 1024:.1f} KB)')

Saved model artifacts in models/:
 -> scaler.pkl (1.1 KB)
 -> isolation_forest.pkl (801.4 KB)
 -> logistic_regression.pkl (0.9 KB)
 -> random_forest.pkl (3842.6 KB)


## Notebook Explanation

### What was done
- Split 284,807 transactions chronologically (80% train = 227,845 txns, 20% test = 56,962 txns) to avoid data leakage.
- Fit StandardScaler strictly on the training set.
- Trained baseline Logistic Regression (`class_weight='balanced'`).
- Trained Random Forest Classifier (`n_estimators=100`, `max_depth=12`, `class_weight='balanced'`).
- Evaluated models on Precision, Recall, F1-Score, PR-AUC, False Positives per 1000 txns, and Business Cost.
- Serialized model artifacts to `models/`.

### Why it was done
Supervised classifiers form the primary machine learning scoring component (0.5 weight) in the hybrid fraud detection pipeline. The offline evaluation establishes baseline detection accuracy and operational cost.

### Input
- Raw dataset from `data/raw/creditcard.csv`.

### Output
- `models/logistic_regression.pkl`
- `models/random_forest.pkl`
- `models/scaler.pkl`
- Quantitative benchmark evaluation table.

### Connection to next stage
The trained supervised models are complemented by unsupervised anomaly detection in `04_isolation_forest_anomaly_detection.ipynb` to identify previously unseen fraud patterns.

### Implementation decisions
- PR-AUC and F1-score were prioritized over accuracy due to extreme class imbalance (0.173% fraud).
- Cost matrix assumption ($500 per missed fraud FN, $15 per false positive alert FP) was applied to quantify business impact.